In [ ]:
import requests
import random
import time

def get_solved_problems(handle):
    """
    Fetches the set of solved problem IDs for a given Codeforces handle.
    """
    try:
        url = f"https://codeforces.com/api/user.status?handle={handle}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if data.get('status') != 'OK':
            print(f"Error fetching data for handle '{handle}': {data.get('comment', 'Unknown error')}")
            return None

        solved_problems = set()
        for submission in data.get('result', []):
            if submission.get('verdict') == 'OK':
                problem = submission.get('problem', {})
                contest_id = problem.get('contestId')
                index = problem.get('index')
                if contest_id and index:
                    problem_id = f"{contest_id}{index}"
                    solved_problems.add(problem_id)

        return solved_problems

    except requests.exceptions.RequestException as e:
        print(f"API request error: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

def get_recent_contest_ids(days=365):
    """
    Fetches the IDs of all finished contests from the last number of days.
    This is a more reliable way to determine "recent" problems.
    """
    try:
        url = "https://codeforces.com/api/contest.list"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if data.get('status') != 'OK':
            print(f"Contest list fetch error: {data.get('comment', 'Unknown error')}")
            return None

        one_year_ago = time.time() - (days * 24 * 60 * 60)
        recent_contest_ids = set()

        for contest in data.get('result', []):
            if contest.get('phase') == 'FINISHED' and contest.get('startTimeSeconds', 0) >= one_year_ago:
                recent_contest_ids.add(contest.get('id'))

        return recent_contest_ids

    except requests.exceptions.RequestException as e:
        print(f"API request error while fetching contests: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while fetching contests: {e}")
        return None

def get_problems_from_contests(recent_contest_ids, min_diff, max_diff):
    """
    Fetches all problems and filters them based on a given set of contest IDs and difficulty range.
    """
    try:
        url = "https://codeforces.com/api/problemset.problems"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if data.get('status') != 'OK':
            print(f"Problemset fetch error: {data.get('comment', 'Unknown error')}")
            return None

        all_problems = data.get('result', {}).get('problems', [])

        # Filter problems that are in the recent_contest_ids set and match the difficulty
        filtered_problems = [
            p for p in all_problems
            if p.get('contestId') in recent_contest_ids and
               'rating' in p and
               min_diff <= p['rating'] <= max_diff
        ]

        return filtered_problems

    except requests.exceptions.RequestException as e:
        print(f"API request error while fetching problemset: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred while fetching the problemset: {e}")
        return None

def main():
    """
    Main function to execute the problem-fetching and selection process.
    """
    # handle = input("Enter your Codeforces handle: ").strip()
    # if not handle:
    #     print("Handle cannot be empty.")
    #     return

    # --- You can adjust these parameters ---
    handle = "saimur"
    min_difficulty = 800
    max_difficulty = 1000
    last_n_days = 100
    num_problems_to_suggest = 3
    # ------------------------------------

    print(f"\n🔍 Fetching solved problems for '{handle}'...")
    solved_problems = get_solved_problems(handle)
    if solved_problems is None:
        print("❌ Error fetching user data. Please check your handle or try again later.")
        return
    print(f"✅ Found {len(solved_problems)} solved problems.")

    print("\n🔍 Fetching recent contests from the last year...")
    recent_contests = get_recent_contest_ids(last_n_days)
    if recent_contests is None:
        print("❌ Error fetching recent contests. Please try again later.")
        return
    print(f"✅ Found {len(recent_contests)} recent contests.")

    print(f"\n🔍 Fetching problems with difficulty {min_difficulty}–{max_difficulty} from these contests...")
    available_problems = get_problems_from_contests(recent_contests, min_difficulty, max_difficulty)
    if available_problems is None:
        print("❌ Error fetching the problemset. Please try again later.")
        return
    print(f"✅ Found {len(available_problems)} problems in that range.")

    # Filter out problems the user has already solved
    unsolved_problems = [
        p for p in available_problems
        if f"{p.get('contestId')}{p.get('index')}" not in solved_problems
    ]
    print(f"🧩 Found {len(unsolved_problems)} unsolved problems in that range.")

    if not unsolved_problems:
        print("\n🎉 You've solved all recent problems in this difficulty range, or none are available. Great job! Consider increasing the difficulty.")
        return

    # Select a random sample of unsolved problems
    selected_problems = random.sample(unsolved_problems, min(num_problems_to_suggest, len(unsolved_problems)))

    print(f"\n📚 Here are {len(selected_problems)} random unsolved problems for you:\n")
    for i, problem in enumerate(selected_problems, 1):
        link = f"https://codeforces.com/problemset/problem/{problem.get('contestId')}/{problem.get('index')}"
        print(f"{i}. {problem.get('name', 'N/A')}")
        print(f"   🔗 {link}\n")

if __name__ == "__main__":
    main()


🔍 Fetching solved problems for 'saimur'...
✅ Found 449 solved problems.

🔍 Fetching recent contests from the last year...
✅ Found 24 recent contests.

🔍 Fetching problems with difficulty 800–1000 from these contests...
✅ Found 31 problems in that range.
🧩 Found 28 unsolved problems in that range.

📚 Here are 3 random unsolved problems for you:

1. Greedy Grid
   🔗 https://codeforces.com/problemset/problem/2122/A

2. Energy Crystals
   🔗 https://codeforces.com/problemset/problem/2111/A

3. Square Pool
   🔗 https://codeforces.com/problemset/problem/2120/B

